In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from loguru import logger
from torch.utils.data import Dataset

from data_preprocessing.utils.data_handling import load_parquet_table

import hvplot.pandas


In [ ]:
notebook_path = Path.cwd()
project_root = notebook_path.parents[0]
dataset_ts_path: Path = project_root / Path(
    "datasets/raw_and_shitty/dataset_ts_only.parquet")

In [ ]:

df_dataset_ts = load_parquet_table(parquet_path=dataset_ts_path, backend="numpy_nullable", verbose=False)
df_dataset_ts.head()

In [ ]:

patient_ids = df_dataset_ts["patientunitstayid"].unique().tolist()
df_dataset_cont_all = df_dataset_ts.loc[:,
                      ["patientunitstayid", "time_offset", "heartrate", "sao2", "temperature", "st1",
                       "sepsis_present", "sepsis_patient"]]

df_dataset_cont = df_dataset_cont_all[df_dataset_cont_all["patientunitstayid"].isin(patient_ids[0:29])].copy()

In [ ]:
patient_ids_sepsis = df_dataset_cont_all.loc[df_dataset_cont_all["sepsis_patient"] == True][
    "patientunitstayid"].unique().tolist()

df_dataset_cont_sepsis = df_dataset_cont_all[
    df_dataset_cont_all["patientunitstayid"].isin(patient_ids_sepsis[0:29])].copy()
df_dataset_cont_sepsis.head()

In [ ]:
target = "sepsis_present"

features = list(df_dataset_cont.columns.difference([target]))

df_train = df_dataset_cont.loc[df_dataset_cont["patientunitstayid"].isin(patient_ids[0:20])]
df_test = df_dataset_cont.loc[df_dataset_cont["patientunitstayid"].isin(patient_ids[21:29])]

logger.debug(f"{len(df_test) / len(df_dataset_cont) = }")


In [ ]:
df_dataset_cont.head()

In [ ]:
patient_ids = df_dataset_cont["patientunitstayid"].unique().tolist()

In [ ]:
df_sample = df_dataset_cont[df_dataset_cont["patientunitstayid"] == patient_ids[0]]

In [ ]:
df_sample["time_offset"].describe()

In [ ]:
input_window_length = pd.to_timedelta(6, unit="h")
input_window_length

In [ ]:
df_sample["time_offset"].max() / input_window_length

The last input window length chunk might be to short to be useful, we will discard such samples.
This won't prevent us from taking advantage of all the sepsis labels, we consider already only sepsis patients that get the condition within 168 h while the maximum length of our control samples ist 192 h.

In [ ]:
number_of_windows_for_input_window_size = math.floor(df_sample["time_offset"].max() / input_window_length)
number_of_windows_for_input_window_size

In [ ]:
#current_sample = 1

In [ ]:
#sample_1_sepsis_status_horizon_time = current_sample * prediction_horizon
#sample_1_sepsis_status_horizon_time

In [ ]:
df_sample_1 = df_sample[df_sample["time_offset"] < input_window_length]
df_sample_1

In [ ]:
df_sample_1.describe()

In [ ]:
df_sample_1.head()

## Aligning the time series to a 5 min grid

In [ ]:
time_grid_freq: str = "5min"

Determining the time grid floor and ceiling, to properly represent the smallest and biggest available time stamp of the time series, so we can align the TS without dropping any values

In [ ]:
df_sample_1["time_offset"].min()

In [ ]:
time_grid_floor = df_sample_1["time_offset"].min().floor(time_grid_freq)
time_grid_floor

In [ ]:
df_sample_1["time_offset"].max()

In [ ]:
time_grid_ceiling = df_sample_1["time_offset"].max().ceil(time_grid_freq)
time_grid_ceiling

In [ ]:
df_grid_sample = pd.DataFrame(
    data={"time_offset": pd.timedelta_range(start=time_grid_floor, end=time_grid_ceiling, freq="5min"), })

In [ ]:
df_grid_sample

In [ ]:
df_sample_1_gridded = pd.merge_asof(left=df_grid_sample, right=df_sample_1, on="time_offset", direction="forward")


In [ ]:
df_sample_1_gridded

Note! The number of grid ticks (5 min frequency) can be less than the original, the sampling time of individual tables or rather measurements is not synchronized, so multiple time steps might be aligned/merged on the same grid tick -> So this is not a problem


### Shifting the target variable to reframe the prediction as a classification at the end of the input window

In [ ]:
prediction_horizon = pd.to_timedelta(2, unit="h")
prediction_horizon

In [ ]:
prediction_target_shifted = "sepsis_present_shifted"

prediction_horizon_time_steps_number = int(prediction_horizon / pd.to_timedelta("5min"))

prediction_horizon_time_steps_number

In [ ]:
df_sample_1_gridded[prediction_target_shifted] = df_sample_1_gridded["sepsis_present"].shift(
    -prediction_horizon_time_steps_number)

In [ ]:
df_sample_1_gridded["sepsis_present_shifted_filled"] = df_sample_1_gridded["sepsis_present_shifted"].ffill()

In [ ]:
df_sample_1_gridded

In [ ]:
def align_to_grid(df_group: pd.DataFrame.groupby, time_grid_freq: str = "5min") -> pd.DataFrame:
    time_grid_floor = df_group["time_offset"].min().floor(time_grid_freq)
    time_grid_ceiling = df_group["time_offset"].max().ceil(time_grid_freq)

    df_grid_only = pd.DataFrame(
        data={"time_offset": pd.timedelta_range(start=time_grid_floor, end=time_grid_ceiling, freq=time_grid_freq)})

    df_group_gridded = pd.merge_asof(left=df_grid_only, right=df_group, on="time_offset", direction="forward")

    return df_group_gridded

The column has been shifted and resulting forward NAs forward filled with the last know value 

In [ ]:
def shift_and_fill_prediction_target(df_group: pd.DataFrame.groupby, prediction_horizon_h: int = 4,
                                     time_grid_freq: str = "5min") -> pd.DataFrame:
    prediction_horizon = pd.to_timedelta(prediction_horizon_h, unit="h")
    prediction_target_shifted = "sepsis_present_shifted"

    prediction_horizon_time_steps_number = int(prediction_horizon / pd.to_timedelta(time_grid_freq))

    df_group[prediction_target_shifted] = df_group["sepsis_present"].shift(
        -prediction_horizon_time_steps_number)

    df_group["sepsis_present_shifted_filled"] = df_group["sepsis_present_shifted"].ffill()

    return df_group

In [ ]:
df_dataset_cont_gridded = df_dataset_cont.groupby("patientunitstayid", group_keys=False).apply(
    lambda x: align_to_grid(x)).reset_index(drop=True)
df_dataset_cont_gridded.head()


In [ ]:
df_dataset_cont

In [ ]:
df_dataset_cont_gridded

In [ ]:
df_dataset_cont_shifted = df_dataset_cont_gridded.groupby("patientunitstayid", group_keys=False)[
    df_dataset_cont_gridded.columns].apply(lambda x: shift_and_fill_prediction_target(x)).reset_index(drop=True)

In [ ]:
df_dataset_cont_shifted

-> LGMT

Sanity check with sepsis patient data 

In [ ]:
df_dataset_cont_sepsis_gridded = df_dataset_cont_sepsis.groupby("patientunitstayid", group_keys=False)[
    df_dataset_cont_sepsis.columns].apply(lambda x: align_to_grid(x)).reset_index(drop=True)
df_dataset_cont_sepsis_gridded_shifted = df_dataset_cont_sepsis_gridded.groupby("patientunitstayid", group_keys=False)[
    df_dataset_cont_sepsis_gridded.columns].apply(lambda x: shift_and_fill_prediction_target(x)).reset_index(drop=True)


In [ ]:
df_dataset_cont_sepsis_gridded_shifted

LGTM as well


In [ ]:
df_dataset_sepsis_check = df_dataset_cont_sepsis[["patientunitstayid", "time_offset", "heartrate", "sepsis_present"]]
df_dataset_sepsis_shifted_check = df_dataset_cont_sepsis_gridded_shifted[
    ["patientunitstayid", "time_offset", "heartrate", "sepsis_present", "sepsis_present_shifted_filled"]]


In [ ]:
df_dataset_sepsis_check.groupby(by="patientunitstayid").get_group(patient_ids_sepsis[0]).hvplot(x="time_offset",
                                                                                                y="heartrate")

In [ ]:
df_dataset_sepsis_shifted_check.groupby(by="patientunitstayid").get_group(patient_ids_sepsis[0]).hvplot(x="time_offset",
                                                                                                        y=[
                                                                                                            "sepsis_present",
                                                                                                            "sepsis_present_shifted_filled"])

### Slicing or chunking the dataset such that we can feed the samples via an multiindex to the dataloader

#### Calculating the number of samples for a given patient time series

In [ ]:
df_sample_chunking = df_dataset_sepsis_shifted_check.groupby(by="patientunitstayid").get_group(patient_ids_sepsis[0])

In [ ]:
chunking_sample_min = df_sample_chunking["time_offset"].min()
chunking_sample_max = df_sample_chunking["time_offset"].max()

print(chunking_sample_min)
print(chunking_sample_max)


In [ ]:
chunking_sample_entire_length = abs(chunking_sample_max - chunking_sample_min)
chunking_sample_entire_length

In [ ]:
number_of_chunks_sample = math.floor(chunking_sample_entire_length / input_window_length)
number_of_chunks_sample

In [ ]:
df_sample_chunking.head()

In [ ]:
pytorch_sample_counter = 0

In [ ]:
list_df_chunked = list()

for i in range(number_of_chunks_sample):
    chunk_time_lower = input_window_length * i
    chunk_time_upper = input_window_length * (i + 1)

    logger.debug(f"{i = }, {chunk_time_lower = }, {chunk_time_upper = }")
    # chunk_time = df_sample_chunking.loc[df_sample_chunking["time_offset"] == (input_window_length * i):]
    df_chunk_of_time_range = df_sample_chunking.loc[(df_sample_chunking["time_offset"] >= chunk_time_lower) & (
            df_sample_chunking["time_offset"] <= chunk_time_upper)]

    df_chunk_of_time_range = df_chunk_of_time_range.copy()
    df_chunk_of_time_range["patient_sample_count"] = i

    pytorch_sample_counter += 1

    df_chunk_of_time_range.set_index(["patient_sample_count"], inplace=True)

    list_df_chunked.append(df_chunk_of_time_range)
    logger.debug(f"{df_chunk_of_time_range = }")




In [ ]:
pytorch_sample_counter

In [ ]:
list_df_chunked

In [ ]:
list_df_chunked[0]

In [ ]:
list_df_chunked[5]

In [ ]:
# Todo: Einen Assert reinhauen, dass die chunks wieder zusammengesetzt das input dataframe ergeben (da muss man aber beachten, dass der input nicht komplett hergenommen wird)

In [ ]:
list_df_chunked

In [ ]:
from functools import reduce

df_sample_chunked_dataloader_index = reduce(lambda left, right: pd.concat([left, right]),
                                            list_df_chunked)  #.set_index("patient_sample_count")
df_sample_chunked_dataloader_index

In [ ]:
df_sample_chunked_dataloader_index["patient_sample_count"] = df_sample_chunked_dataloader_index.reset_index()[
    "patient_sample_count"].drop_duplicates(keep="last")

In [ ]:
df_sample_chunking

In [ ]:
def chunk_df_based_on_time(df_group: pd.DataFrame.groupby, total_sample_counter):
    chunking_sample_min = df_group["time_offset"].min()
    chunking_sample_max = df_group["time_offset"].max()

    chunking_sample_entire_length = abs(chunking_sample_max - chunking_sample_min)

    number_of_chunks_sample = math.floor(chunking_sample_entire_length / input_window_length)

    list_df_individual_chunks = []

    for i in range(number_of_chunks_sample):
        chunk_time_lower = input_window_length * i
        chunk_time_upper = input_window_length * (i + 1)

        logger.debug(f"{i = }, {chunk_time_lower = }, {chunk_time_upper = }")
        # chunk_time = df_sample_chunking.loc[df_sample_chunking["time_offset"] == (input_window_length * i):]
        df_chunk_of_time_range = df_group.loc[(df_group["time_offset"] >= chunk_time_lower) & (
                df_group["time_offset"] <= chunk_time_upper)]

        df_chunk_of_time_range = df_chunk_of_time_range.copy()
        df_chunk_of_time_range["patient_sample_count"] = total_sample_counter.value()

        total_sample_counter.increment()

        logger.debug(f"{total_sample_counter.value() = }")

        df_chunk_of_time_range = df_chunk_of_time_range.set_index(["patient_sample_count"], inplace=False)

        list_df_individual_chunks.append(df_chunk_of_time_range)
        # logger.debug(f"{df_chunk_of_time_range = }")

        df_group_sampled_tracked = reduce(lambda left, right: pd.concat([left, right]),
                                          list_df_individual_chunks)

    return df_group_sampled_tracked

In [ ]:
df_dataset_sepsis_check

In [ ]:
class SampleCount:
    """Helper class that tracks the total number of samples for overall multiindex"""

    def __init__(self):
        self.total_sample_count: int = 0

    def increment(self):
        self.total_sample_count += 1

    def value(self):
        return self.total_sample_count

In [ ]:
overall_sample_counter = SampleCount()

In [ ]:

def add_overall_sample_idx_to_df(df_for_dataloader):
    df_for_dataloader_indexed = df_for_dataloader.groupby(by="patientunitstayid", group_keys=False)[
        df_for_dataloader.columns].apply(
        lambda x: chunk_df_based_on_time(df_group=x, total_sample_counter=overall_sample_counter))

    return df_for_dataloader_indexed


In [ ]:
df_dataset_cont_sepsis_gridded_shifted

In [ ]:
df_dataloader = add_overall_sample_idx_to_df(df_for_dataloader=df_dataset_cont_sepsis_gridded_shifted)


In [ ]:
df_dataloader

In [ ]:
df_dataloader.index

In [ ]:
df_dataloader.describe()

-> LGTM